<div class="alert alert-block alert-success">

<b>CODING BACKPROPAGATION FROM SCRATCH: ON A SINGLE NEURON</b>
</div>

In [2]:
import numpy as np 

# initialize parameters 
inputs =  np.array([1.0, -2.0, 3.0])
weights = np.array([-3.0, -1.0, 2.0])
bias = 1.0
target_output = 0
learning_rate = 0.001

def relu(x):
    return np.maximum(x, 0)

def relu_derivative(x):
    return np.where(x>0, 1.0, 0)

for i in range(20):
    # Forward pass 
    sum = np.dot(inputs, weights) + bias
    output = relu(sum)
    loss = (output - target_output)**2

    # Backward pass
    dloss_doutput = 2 * (output - target_output)
    doutput_drelu = relu_derivative(sum)
    dmul_dwi = inputs
    dmul_dbias = 1.0

    dloss_dwi = dloss_doutput * doutput_drelu * dmul_dwi
    dloss_dbi = dloss_doutput * doutput_drelu * dmul_dbias

    # Update the weights and bias
    weights = weights - learning_rate * dloss_dwi
    bias = bias - learning_rate * dloss_dbi

    # Print the loss for this iteration
    print(f"Iteration {i+ 1}, Loss: {loss}")

print(f"Final Loss {loss}")
print(f"Final Bias {bias}")


Iteration 1, Loss: 36.0
Iteration 2, Loss: 33.872399999999985
Iteration 3, Loss: 31.870541159999995
Iteration 4, Loss: 29.98699217744401
Iteration 5, Loss: 28.21476093975706
Iteration 6, Loss: 26.54726856821742
Iteration 7, Loss: 24.978324995835766
Iteration 8, Loss: 23.502105988581878
Iteration 9, Loss: 22.113131524656684
Iteration 10, Loss: 20.80624545154949
Iteration 11, Loss: 19.576596345362915
Iteration 12, Loss: 18.419619501351963
Iteration 13, Loss: 17.331019988822064
Iteration 14, Loss: 16.306756707482677
Iteration 15, Loss: 15.343027386070442
Iteration 16, Loss: 14.43625446755368
Iteration 17, Loss: 13.583071828521266
Iteration 18, Loss: 12.780312283455652
Iteration 19, Loss: 12.024995827503426
Iteration 20, Loss: 11.314318574097976
Final Loss 11.314318574097976
Final Bias 0.8175177371706989


<div class="alert alert-block alert-success">
<b>BACKPROPAGATION </b>
</div>

<div class="alert alert-block alert-warning">
GRADIENTS OF THE LOSS WITH RESPECT TO WEIGHTS
</div>

In [3]:
import numpy as np
# Passed-in gradient from the next layer
# for the purpose of this example we're going to use
# an array of an incremental gradient values
dvalues = np.array([[1., 1., 1.],
 [2., 2., 2.],
 [3., 3., 3.]])
# We have 3 sets of inputs - samples
inputs = np.array([[1, 2, 3, 2.5],
 [2., 5., -1., 2],
 [-1.5, 2.7, 3.3, -0.8]])
# sum weights of given input
# and multiply by the passed-in gradient for this neuron

dweights = np.dot(inputs.T, dvalues)
print(dweights)

[[ 0.5  0.5  0.5]
 [20.1 20.1 20.1]
 [10.9 10.9 10.9]
 [ 4.1  4.1  4.1]]


<div class="alert alert-block alert-warning">
GRADIENTS OF THE LOSS WITH RESPECT TO BIASES
</div>

In [4]:
import numpy as np
# Passed-in gradient from the next layer
# for the purpose of this example we're going to use
# an array of an incremental gradient values
dvalues = np.array([[1., 1., 1.],
 [2., 2., 2.],
 [3., 3., 3.]])
# One bias for each neuron
# biases are the row vector with a shape (1, neurons)
biases = np.array([[2, 3, 0.5]])
# dbiases - sum values, do this over samples (first axis), keepdims
# since this by default will produce a plain list -

dbiases = np.sum(dvalues, axis= 0, keepdims= True)
print(dbiases)


[[6. 6. 6.]]


<div class="alert alert-block alert-warning">
GRADIENTS OF THE LOSS WITH RESPECT TO INPUTS
</div>

In [5]:
import numpy as np
# Passed-in gradient from the next layer
# for the purpose of this example we're going to use
# an array of an incremental gradient values
dvalues = np.array([[1., 1., 1.],
 [2., 2., 2.],
 [3., 3., 3.]])
# We have 3 sets of weights - one set for each neuron
# we have 4 inputs, thus 4 weights
# recall that we keep weights transposed
weights = np.array([[0.2, 0.8, -0.5, 1],
 [0.5, -0.91, 0.26, -0.5],
 [-0.26, -0.27, 0.17, 0.87]]).T
# sum weights of given input
# and multiply by the passed-in gradient for this neuron

dinputs = np.dot(dvalues, weights.T)
print(dinputs)

[[ 0.44 -0.38 -0.07  1.37]
 [ 0.88 -0.76 -0.14  2.74]
 [ 1.32 -1.14 -0.21  4.11]]


## Definition of Layer with backpropagation

In [6]:
class Layer:
    def __init__(self, row, cols):
        self.weights = 0.01 * np.random.randn(row, cols)
        self.bias = np.zeros((1, cols))

    def forward(self, inputs):
        self.x = inputs
        self.output = np.dot(self.x, self.weights) + self.bias
        return self.output

    def backward(self, dvalues):
        self.dL_dz = dvalues
        self.dweights = np.dot(self.x.T, self.dL_dz  )
        self.dbias = np.sum(self.dL_dz, axis = 0, keepdims= True)
        self.dinputs = np.dot(self.dL_dz, self.weights.T)

## Definition of ReLU activation function with backpropagation


In [7]:
class relu:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        return self.output

    def backward(self, dvalues):
        self.dinputs = dvalues.copy()
        self.dinputs[self.inputs <= 0] = 0

## Definition of Categorical Cross Entropy Loss

In [8]:
class CrossEntropyLoss:
    def forward(self, y_pred, y_true):
        y_pred_cliped = np.clip(y_pred, 1e-5, 1-1e-5 )
        # check for one hot encoding 
        if (len(y_true.shape)== 1):
            # Not One hot encoded 
            y_true = np.eye(len(y_pred[0]))[y_true]

        
        correct_confidences = np.sum(y_true * np.log(y_pred_cliped), axis=1, keepdims= True)

        neg_loss = - correct_confidences
        avg_loss = np.mean(neg_loss)
        return avg_loss

    def backward(self, dvalues , y_true):
        if (len(y_true.shape)== 1):
            y_true = np.eye(len(dvalues[0]))[y_true]

        dvalues = np.clip(dvalues, 1e-5, 1-1e-5)
        
        self.dinputs = - y_true/dvalues
        self.dinputs = self.dinputs/(len(dvalues))

        
        

In [9]:
# Softmax activation
class Activation_Softmax:
 # Forward pass
 def forward(self, inputs):
 # Get unnormalized probabilities
  exp_values = np.exp(inputs - np.max(inputs, axis=1, keepdims=True))
 # Normalize them for each sample
  probabilities = exp_values / np.sum(exp_values, axis=1,keepdims=True)
  self.output = probabilities

## Definition of Softmax loss backpropagation

In [10]:
class Activation_Softmax_crossEntropy_loss:
    def __init__(self):
        self.softmax = Activation_Softmax()
        self.cross = CrossEntropyLoss()

    def forward(self, inputs, y_true):
        self.softmax.forward(inputs)
        self.outputs = self.cross.forward(self.softmax.output, y_true)
        return self.outputs

    def backward(self, predicted , y_true):
        samples = len(predicted)
        # if the y_true is one hot encoded
        if(len(y_true.shape) == 2):
            y_true = np.argmax(y_true, axis = 1)
        
        self.dinputs = predicted.copy()
        self.dinputs[range(len(predicted)), y_true] -= 1
        self.dinputs = self.dinputs/samples

In [11]:
softmax_outputs = np.array([
    [0.7, 0.1, 0.2],
    [0.1, 0.5, 0.4],
    [0.02, 0.9, 0.08]
])

y_true = np.array([0, 1, 1])

softmax = Activation_Softmax_crossEntropy_loss()
softmax.backward(softmax_outputs, y_true)
dvalues = softmax.dinputs
print("Gradient: combined loss and activation func:", dinputs)


Gradient: combined loss and activation func: [[ 0.44 -0.38 -0.07  1.37]
 [ 0.88 -0.76 -0.14  2.74]
 [ 1.32 -1.14 -0.21  4.11]]


# ASSAMBLE EVERYTHING TO MAKE A NEURAL NETWORK

In [12]:
from nnfs.datasets import spiral_data


<img src = 'attachments/full_nn.jpeg' alt = 'dense layer example' width = 'auto' height = 'auto'>

In [13]:

X, y = spiral_data(samples=100, classes=3)

# first Layer 2 inputs -> 3 neuron
dense1 = Layer(2, 3)

# ReLU layer
activation1 = relu()

# secend dense layer 3 inputs -> 3 neuron
dense2 = Layer(3, 3)


# softmax and categorical cross entropy loss
loss = Activation_Softmax_crossEntropy_loss()

dense1.forward(X)
activation1.forward(dense1.output)
dense2.forward(activation1.output)
loss.forward(dense2.output, y)

print(f'loss = {loss.outputs}')

predictions = np.argmax(loss.softmax.output, axis = 1)
if len(y.shape) ==2:
    y = np.argmax(y, axis = 1)

acc = np.mean(predictions == y)
print(f"Accuracy= {acc}")

# Backward pass 
loss.backward(loss.softmax.output, y)
dense2.backward(loss.dinputs)
activation1.backward(dense2.dinputs)
dense1.backward(activation1.dinputs)

# Print gradients
print(dense1.dweights)
print(dense1.dbias)
print(dense2.dweights)
print(dense2.dbias)



loss = 1.098603706755002
Accuracy= 0.4166666666666667
[[ 3.51955628e-05  5.95979441e-05 -5.94632693e-05]
 [-2.18969398e-04  3.43878846e-04  2.24972401e-04]]
[[-0.00062593 -0.00070527  0.00040587]]
[[ 1.13993430e-04 -1.30761516e-04  1.67680864e-05]
 [-1.09838312e-04  3.40469176e-04 -2.30630864e-04]
 [ 4.13973812e-05 -8.41251856e-06 -3.29848626e-05]]
[[-9.47680703e-08 -1.15527536e-05  1.16475217e-05]]


## Gradient Descent Optimizer implementation


In [24]:
class Optimization_SGD:
    def __init__(self, learning_rate):
        self.learning_rate = learning_rate

    def update_para(self, layer):
        layer.weights += - layer.dweights * self.learning_rate
        layer.bias += - layer.dbias * self.learning_rate


In [25]:
X, y = spiral_data(samples = 100, classes = 3)

# first Layer 2 inputs -> 64 neuron
dense1 = Layer(2, 64)

# ReLU layer
activation1 = relu()

# secend dense layer 64 inputs -> 3 neuron
dense2 = Layer(64, 3)


# softmax and categorical cross entropy loss
loss = Activation_Softmax_crossEntropy_loss()
optimizer = Optimization_SGD(learning_rate= 1.0)

for epoch in range(10001):
    
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    data_loss = loss.forward(dense2.output, y)


    predictions = np.argmax(loss.softmax.output, axis = 1)
    if len(y.shape) ==2:
        y_true_1d = np.argmax(y, axis = 1)
    else:
        y_true_1d = y

    acc = np.mean(predictions == y_true_1d)
    if (epoch % 100 == 0):
        print(f"iteration = {epoch}, Accuracy= {acc}, loss = {data_loss}")

    # Backward pass 
    loss.backward(loss.softmax.output, y)
    dense2.backward(loss.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # Optimizer: Gradient Descent
    optimizer.update_para(layer= dense1)
    optimizer.update_para(layer = dense2)




iteration = 0, Accuracy= 0.24333333333333335, loss = 1.0986479124623791
iteration = 100, Accuracy= 0.39666666666666667, loss = 1.0794072571154003
iteration = 200, Accuracy= 0.42333333333333334, loss = 1.071721328870253
iteration = 300, Accuracy= 0.4033333333333333, loss = 1.0703370252896751
iteration = 400, Accuracy= 0.4, loss = 1.0696076883656354
iteration = 500, Accuracy= 0.39666666666666667, loss = 1.0687929102568574
iteration = 600, Accuracy= 0.41333333333333333, loss = 1.067375517459968
iteration = 700, Accuracy= 0.41333333333333333, loss = 1.0648412679413362
iteration = 800, Accuracy= 0.4266666666666667, loss = 1.0609908961504875
iteration = 900, Accuracy= 0.44666666666666666, loss = 1.0546496393299158
iteration = 1000, Accuracy= 0.3433333333333333, loss = 1.0563905758582406
iteration = 1100, Accuracy= 0.37666666666666665, loss = 1.0452176858084432
iteration = 1200, Accuracy= 0.4, loss = 1.0367548600114158
iteration = 1300, Accuracy= 0.41333333333333333, loss = 1.0278494788337853

## Gradient Descent Optimizer with decaying learning_rate

In [26]:
class Optimization_SGD:
    def __init__(self, learning_rate = 1., decay = 0.):
        self.learning_rate = learning_rate
        self.decay = decay
        self.current_learning_rate = self.learning_rate
        self.iteration = 0

    def pre_update(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate/(1. + self.decay * self.iteration)
    
    def update_para(self, layer):
        layer.weights += - self.current_learning_rate * layer.dweights
        layer.bias += -self.current_learning_rate * layer.dbias

    def post_update(self):
        self.iteration += 1

In [27]:


dense1 = Layer(2, 64)
activation1 = relu()
dense2 = Layer(64, 3)
loss_activation = Activation_Softmax_crossEntropy_loss()
optimizer = Optimization_SGD(learning_rate= 1.0 , decay = 0.001)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    loss = loss_activation.forward(dense2.output, y)

    predictions = np.argmax(loss_activation.softmax.output , axis = 1)
    if (len(y.shape) == 2):
        y_true_1d = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y_true_1d)

    if (epoch%100 == 0):
        print(f"iteration = {epoch}, accuracy = {accuracy}, loss = {loss}, lr = {optimizer.current_learning_rate}")
    
    # calculating gradients 
    loss_activation.backward(loss_activation.softmax.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # optimize the network
    optimizer.pre_update()
    optimizer.update_para(dense1)
    optimizer.update_para(dense2)
    optimizer.post_update()


iteration = 0, accuracy = 0.32, loss = 1.0985747735596674, lr = 1.0
iteration = 100, accuracy = 0.4, loss = 1.0789531443939036, lr = 0.9099181073703367
iteration = 200, accuracy = 0.4066666666666667, loss = 1.072152448994514, lr = 0.8340283569641367
iteration = 300, accuracy = 0.39666666666666667, loss = 1.0707612746712294, lr = 0.7698229407236336
iteration = 400, accuracy = 0.39666666666666667, loss = 1.0700877289294617, lr = 0.7147962830593281
iteration = 500, accuracy = 0.4, loss = 1.069410789993943, lr = 0.66711140760507
iteration = 600, accuracy = 0.39666666666666667, loss = 1.0686881750491743, lr = 0.6253908692933083
iteration = 700, accuracy = 0.4033333333333333, loss = 1.0674885354816017, lr = 0.5885815185403178
iteration = 800, accuracy = 0.4066666666666667, loss = 1.0662978768758307, lr = 0.5558643690939411
iteration = 900, accuracy = 0.4033333333333333, loss = 1.0649474292380334, lr = 0.526592943654555
iteration = 1000, accuracy = 0.4066666666666667, loss = 1.063306924107328

## Gradient Descent with momentum 

In [28]:
class Optimization_SGD:
    def __init__(self,  learning_rate =1. , decay = 0.0, momentum = 0.):
        self.learning_rate = learning_rate
        self.decay = decay
        self.current_learning_rate = self.learning_rate
        self.momentum = momentum 
        self.iterations = 0

    def pre_updates(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate/(1 + self.iterations * self.decay)

    
    def update_params(self, layer):
        if self.momentum:
            
            if not hasattr(layer, "weight_momentums"):
                layer.weight_momentums = np.zeros_like(layer.weights)
                layer.bias_momentums = np.zeros_like(layer.bias)

            
            weight_updates = self.momentum * layer.weight_momentums - self.current_learning_rate * layer.dweights
            layer.weight_momentums = weight_updates

            bias_updates = self.momentum * layer.bias_momentums - self.current_learning_rate * layer.dbias
            layer.bias_momentums = bias_updates

        else:
            weight_updates = - self.current_learning_rate * layer.dweights
            bias_updates = - self.current_learning_rate * layer.bias

        layer.weights += weight_updates
        layer.bias += bias_updates

    def post_updates(self):
        self.iterations +=1

    

In [29]:
dense1 = Layer(2, 64)
activation1 = relu()
dense2 = Layer(64, 3)
loss_activation = Activation_Softmax_crossEntropy_loss()
optimizer = Optimization_SGD(learning_rate= 1.0 , decay = 0.001, momentum=0.9)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    loss = loss_activation.forward(dense2.output, y)

    predictions = np.argmax(loss_activation.softmax.output , axis = 1)
    if (len(y.shape) == 2):
        y_true_1d = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y_true_1d)

    if (epoch%100 == 0):
        print(f"iteration = {epoch}, accuracy = {accuracy}, loss = {loss}, lr = {optimizer.current_learning_rate}")
    
    # calculating gradients 
    loss_activation.backward(loss_activation.softmax.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # optimize the network
    optimizer.pre_updates()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_updates()

iteration = 0, accuracy = 0.3333333333333333, loss = 1.0986101214945756, lr = 1.0
iteration = 100, accuracy = 0.39, loss = 1.0384336423294438, lr = 0.9099181073703367
iteration = 200, accuracy = 0.42, loss = 0.9846444364352449, lr = 0.8340283569641367
iteration = 300, accuracy = 0.5333333333333333, loss = 0.8652830124529911, lr = 0.7698229407236336
iteration = 400, accuracy = 0.6033333333333334, loss = 0.7839537653218035, lr = 0.7147962830593281
iteration = 500, accuracy = 0.65, loss = 0.7349105520532107, lr = 0.66711140760507
iteration = 600, accuracy = 0.6566666666666666, loss = 0.7030289907481656, lr = 0.6253908692933083
iteration = 700, accuracy = 0.6633333333333333, loss = 0.679750021031269, lr = 0.5885815185403178
iteration = 800, accuracy = 0.6666666666666666, loss = 0.6647973965250042, lr = 0.5558643690939411
iteration = 900, accuracy = 0.6633333333333333, loss = 0.6538072088318561, lr = 0.526592943654555
iteration = 1000, accuracy = 0.6666666666666666, loss = 0.644080280685109

## Adagrad optimizer implementation 

In [32]:
class optimizer_adagrad:
    def __init__(self, learning_rate = 1., decay = 0., epsilon = 1e-7):
        self.learning_rate = learning_rate
        self.decay = decay 
        self.epsilon = epsilon 
        self.interations = 0
        self.current_learning_rate = self.learning_rate

    def pre_update(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate/(1 + self.decay * self.interation)

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.bias)

        layer.weight_cache += layer.dweights**2
        layer.bias_cache += layer.dbias**2

        weights_update = - self.current_learning_rate * layer.dweights/(np.sqrt(layer.weight_cache + self.epsilon))
        bias_update = -self.current_learning_rate * layer.dbias/(np.sqrt(layer.bias_cache + self.epsilon))

        layer.weights += weights_update
        layer.bias +=bias_update
    
    def post_update(self):
        self.interations +=1


In [33]:
dense1 = Layer(2, 64)
activation1 = relu()
dense2 = Layer(64, 3)
loss_activation = Activation_Softmax_crossEntropy_loss()
optimizer = Optimization_SGD(learning_rate= 1.0 , decay = 0.001)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    loss = loss_activation.forward(dense2.output, y)

    predictions = np.argmax(loss_activation.softmax.output , axis = 1)
    if (len(y.shape) == 2):
        y_true_1d = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y_true_1d)

    if (epoch%100 == 0):
        print(f"iteration = {epoch}, accuracy = {accuracy}, loss = {loss}, lr = {optimizer.current_learning_rate}")
    
    # calculating gradients 
    loss_activation.backward(loss_activation.softmax.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # optimize the network
    optimizer.pre_updates()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_updates()

iteration = 0, accuracy = 0.29, loss = 1.0986416247711894, lr = 1.0
iteration = 100, accuracy = 0.39, loss = 1.0913981071251855, lr = 0.9099181073703367
iteration = 200, accuracy = 0.4, loss = 1.07355852241949, lr = 0.8340283569641367
iteration = 300, accuracy = 0.41333333333333333, loss = 1.0719141438861781, lr = 0.7698229407236336
iteration = 400, accuracy = 0.4166666666666667, loss = 1.0711469295567464, lr = 0.7147962830593281
iteration = 500, accuracy = 0.41333333333333333, loss = 1.0707577454119912, lr = 0.66711140760507
iteration = 600, accuracy = 0.4066666666666667, loss = 1.070576687917731, lr = 0.6253908692933083
iteration = 700, accuracy = 0.4066666666666667, loss = 1.0704895072847394, lr = 0.5885815185403178
iteration = 800, accuracy = 0.4, loss = 1.0704485025342596, lr = 0.5558643690939411
iteration = 900, accuracy = 0.4, loss = 1.070427035850643, lr = 0.526592943654555
iteration = 1000, accuracy = 0.4, loss = 1.0704133336907655, lr = 0.5002501250625312
iteration = 1100, ac

In [36]:
class optimizer_adagrad:
    def __init__(self, learning_rate = 1., decay = 0., epsilon = 1e-7, rho = 0.999 ):
        self.learning_rate = learning_rate
        self.decay = decay 
        self.epsilon = epsilon 
        self.interations = 0
        self.current_learning_rate = self.learning_rate
        self.rho = rho 

    def pre_update(self):
        if self.decay:
            self.current_learning_rate = self.learning_rate/(1 + self.decay * self.interation)

    def update_params(self, layer):
        if not hasattr(layer, 'weight_cache'):
            layer.weight_cache = np.zeros_like(layer.weights)
            layer.bias_cache = np.zeros_like(layer.bias)

        layer.weight_cache = layer.weight_cache * self.rho + (1 - self.rho)* layer.dweights**2
        layer.bias_cache = layer.bias_cache * self.rho + (1- self.rho)* layer.dbias**2

        weights_update = - self.current_learning_rate * layer.dweights/(np.sqrt(layer.weight_cache + self.epsilon))
        bias_update = -self.current_learning_rate * layer.dbias/(np.sqrt(layer.bias_cache + self.epsilon))

        layer.weights += weights_update
        layer.bias +=bias_update
    
    def post_update(self):
        self.interations +=1

In [37]:
dense1 = Layer(2, 64)
activation1 = relu()
dense2 = Layer(64, 3)
loss_activation = Activation_Softmax_crossEntropy_loss()
optimizer = Optimization_SGD(learning_rate= 1.0 , decay = 0.001)

for epoch in range(10001):
    dense1.forward(X)
    activation1.forward(dense1.output)
    dense2.forward(activation1.output)
    loss = loss_activation.forward(dense2.output, y)

    predictions = np.argmax(loss_activation.softmax.output , axis = 1)
    if (len(y.shape) == 2):
        y_true_1d = np.argmax(y, axis = 1)
    accuracy = np.mean(predictions == y_true_1d)

    if (epoch%100 == 0):
        print(f"iteration = {epoch}, accuracy = {accuracy}, loss = {loss}, lr = {optimizer.current_learning_rate}")
    
    # calculating gradients 
    loss_activation.backward(loss_activation.softmax.output, y)
    dense2.backward(loss_activation.dinputs)
    activation1.backward(dense2.dinputs)
    dense1.backward(activation1.dinputs)

    # optimize the network
    optimizer.pre_updates()
    optimizer.update_params(dense1)
    optimizer.update_params(dense2)
    optimizer.post_updates()

iteration = 0, accuracy = 0.37666666666666665, loss = 1.098583217552725, lr = 1.0
iteration = 100, accuracy = 0.39666666666666667, loss = 1.0911011149151817, lr = 0.9099181073703367
iteration = 200, accuracy = 0.4033333333333333, loss = 1.0731993138183822, lr = 0.8340283569641367
iteration = 300, accuracy = 0.4166666666666667, loss = 1.0713968803384117, lr = 0.7698229407236336
iteration = 400, accuracy = 0.41333333333333333, loss = 1.0707451130950167, lr = 0.7147962830593281
iteration = 500, accuracy = 0.4066666666666667, loss = 1.0705155222921625, lr = 0.66711140760507
iteration = 600, accuracy = 0.4066666666666667, loss = 1.0704222270088357, lr = 0.6253908692933083
iteration = 700, accuracy = 0.4033333333333333, loss = 1.070393732861771, lr = 0.5885815185403178
iteration = 800, accuracy = 0.4, loss = 1.0703774268230761, lr = 0.5558643690939411
iteration = 900, accuracy = 0.4, loss = 1.0703660701000755, lr = 0.526592943654555
iteration = 1000, accuracy = 0.4, loss = 1.0703595799061192